# 04 · Silver → Gold

Code: [src/gold.py](../src/gold.py) · Tài liệu: ARCHITECTURE §4

Gold **không làm sạch gì thêm**: chỉ JOIN fact với dimension rồi GROUP BY bằng Spark SQL. Mỗi bảng
Silver được đăng ký thành một temp view cùng tên.

Đầu vào là Silver trong sandbox (notebook 03). Chưa có thì tự dựng Bronze → Silver.

**An toàn dữ liệu**: landing chỉ **đọc**. Mọi thứ ghi ra nằm trong `notebook/_sandbox/` — **không**
đụng `data/bronze`, `data/silver`, `data/gold`, `output/`. Mỗi notebook chỉ dọn phần sandbox của mình.

Chạy từ trên xuống (Run All). Ô có `assert` là ô **kiểm**: báo đỏ = lệch.

## 0. Chuẩn bị

In [ ]:
import csv, json, logging, shutil, sys
from pathlib import Path

import pandas as pd

# Thư mục gốc dự án = nơi có src/config.py (kernel có thể mở với cwd bất kỳ)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
for p in (ROOT, ROOT / "tests"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src import bronze, config, gold, silver

logging.basicConfig(level=logging.INFO, format="%(levelname)-5s %(name)-7s %(message)s",
                    stream=sys.stdout, force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)
pd.set_option("display.max_columns", 40)

SANDBOX = ROOT / "notebook" / "_sandbox"             # mọi thứ notebook ghi ra nằm ở đây
SB_BRONZE = SANDBOX / "bronze" / "business_valuation"
SB_CONTROL = SANDBOX / "control" / "loaded_batches.csv"
SB_SILVER = SANDBOX / "silver"
SB_GOLD = SANDBOX / "gold"

# Số đo trên 18 lô thật trong tài liệu — để SO, không assert (dữ liệu đổi thì số đổi)
EXPECTED = {"marts": {"mart_vehicle_price": 951, "mart_monthly_trend": 876,
                     "mart_data_quality": 18, "mart_ml_training": 44_748}}

Hàm tiện ích dùng trong notebook này:

In [ ]:
def fresh(*paths: Path) -> None:
    """Xoá các thư mục/file con của sandbox — notebook chỉ dọn phần của mình."""
    for p in paths:
        if SANDBOX not in p.parents:
            raise ValueError(f"{p} nằm ngoài sandbox — không xoá")
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists():
            p.unlink()
    SANDBOX.mkdir(parents=True, exist_ok=True)


def show(df, n: int = 10) -> pd.DataFrame:
    """DataFrame Spark -> pandas để notebook hiển thị thành bảng."""
    return df.limit(n).toPandas()


def compare(actual: dict, expected: dict) -> pd.DataFrame:
    """Bảng so số thực tế với số đo ghi trong tài liệu."""
    keys = list(expected) + [k for k in actual if k not in expected]
    return pd.DataFrame([{"mục": k, "thực tế": actual.get(k, 0), "tài liệu": expected.get(k, "—"),
                          "khớp": actual.get(k, 0) == expected.get(k)} for k in keys])


def ensure_bronze(spark) -> None:
    """Sandbox chưa có Bronze (chưa chạy notebook 02) -> nạp 18 lô."""
    if not SB_BRONZE.exists():
        print("Sandbox chưa có Bronze → nạp 18 lô (như notebook 02)")
        bronze.run(spark, config.LANDING_DIR, SB_BRONZE, SB_CONTROL)


def ensure_silver(spark) -> None:
    """Sandbox chưa có Silver (chưa chạy notebook 03) -> dựng Bronze nếu cần, rồi Silver."""
    ensure_bronze(spark)
    if not (SB_SILVER / "fact_valuation").exists():
        print("Sandbox chưa có Silver → chạy Silver (như notebook 03)")
        silver.run(spark, SB_BRONZE, SB_SILVER)


fresh(SB_GOLD)

In [ ]:
from pyspark.sql import functions as F
from src.spark import get_spark

spark = get_spark("04_gold")
print("Spark", spark.version)

ensure_silver(spark)
active = silver.active_bronze(spark.read.parquet(SB_BRONZE.as_posix()))     # để đối soát
fact = spark.read.parquet((SB_SILVER / "fact_valuation").as_posix())

gold.register_silver(spark, SB_SILVER)
print("Temp view:", [t.name for t in spark.catalog.listTables()])

## 4.1 `mart_vehicle_price` — giá trung vị mới/cũ và tỷ lệ còn lại ρ

In [ ]:
mvp = spark.sql(gold.MART_VEHICLE_PRICE).cache()
rho = mvp.filter(F.col("ty_le_con_lai").isNotNull())
print(f"{mvp.count()} nhóm · đủ cả giá mới & cũ: {rho.count()} · "
      f"trung vị ρ: {rho.select(F.median('ty_le_con_lai')).first()[0]} · "
      f"ρ > 1: {rho.filter('ty_le_con_lai > 1').count()}")
print("Tài liệu: 115 nhóm đủ hai phía · trung vị ρ 0,938 · 13 nhóm ρ > 1")
show(rho.orderBy(F.desc("so_hop_dong")), 10)

## 4.2 `mart_monthly_trend`

In [ ]:
mmt = spark.sql(gold.MART_MONTHLY_TREND)
print(mmt.count(), "dòng")
show(mmt, 10)

## 4.3 `mart_data_quality` — mỗi tháng bao nhiêu dòng vào fact / quarantine, vì luật nào

In [ ]:
dq = spark.sql(gold.mart_data_quality_sql()).toPandas()
rule_cols = [r.lower() for r in silver.RULES]

assert (dq[rule_cols].sum(axis=1) == dq["so_dong_quarantine"]).all(), "Tổng theo luật ≠ quarantine"
assert dq["so_dong_bronze"].sum() == active.count()
dq

## 4.4 `mart_ml_training` — contract đầu ra

1 dòng = 1 hợp đồng (không GROUP BY), split theo tháng duyệt với mốc `ML_TRAIN_END_MONTH`.

In [ ]:
ml = spark.sql(gold.mart_ml_training_sql()).cache()
gold.check_ml_contract(ml)                           # không ném lỗi = đạt contract

split = (ml.groupBy("split").agg(F.count("*").alias("so_dong"), F.min("thang_duyet").alias("tu"),
                                 F.max("thang_duyet").alias("den"),
                                 F.countDistinct("hieu_xe").alias("so_dong_xe"))
           .orderBy(F.desc("split")).toPandas())
overlap = ml.groupBy("thang_duyet").agg(F.countDistinct("split").alias("n")).filter("n > 1").count()

assert ml.count() == fact.count(), "Bảng ML phải đúng 1 dòng / hợp đồng"
assert overlap == 0, "Có tháng nằm ở cả train lẫn test"
print("Mốc:", config.ML_TRAIN_END_MONTH, "· tài liệu: train 36.525 · test 8.223")
split

## 4.5 Contract chặn đầu ra lệch

Cố tình làm lệch bảng theo 4 kiểu; `check_ml_contract` phải chặn **cả 4** trước khi ghi.

In [ ]:
drifts = {
    "thiếu cột split": ml.drop("split"),
    "đổi thứ tự cột": ml.select(*reversed(ml.columns)),
    "tuoi_xe thành bigint": ml.withColumn("tuoi_xe", F.col("tuoi_xe").cast("bigint")),
    "target có null": ml.withColumn("gia_tri_trieu_vnd", F.when(F.col("split") == "train",
                                                                F.col("gia_tri_trieu_vnd"))),
}
outcome = []
for name, bad in drifts.items():
    try:
        gold.check_ml_contract(bad)
        outcome.append({"lệch": name, "kết quả": "LỌT — contract không bắt được!"})
    except ValueError as e:
        outcome.append({"lệch": name, "kết quả": "chặn: " + str(e).splitlines()[0]})
outcome = pd.DataFrame(outcome)

assert outcome["kết quả"].str.startswith("chặn").all()
outcome

## 4.6 Ghi Gold — và chạy lại cho ra đúng kết quả cũ

`csv_dir=None`: không xuất CSV, để `output/` thật không bị ghi đè.

In [ ]:
res_gold = gold.run(spark, SB_SILVER, SB_GOLD, csv_dir=None)
res_gold_again = gold.run(spark, SB_SILVER, SB_GOLD, csv_dir=None)     # full refresh: idempotent

assert res_gold == res_gold_again
compare(res_gold, EXPECTED["marts"])

---
## Dọn dẹp

In [ ]:
spark.stop()
# fresh(SANDBOX / ...)    # xoá phần sandbox của notebook này nếu muốn